# 2.4 · Calculus and automatic differentiation

**Estimated time:** 45–55 minutes · **Prerequisite:** Lesson 2.3 (a weight, a prediction, and an error).

### What you will understand by the end

- A **derivative** describes how an output changes when an input changes.
- A **gradient** collects partial derivatives for several adjustable parameters.
- The **chain rule** tells us how changes flow through a sequence of calculations.
- A **forward pass** computes a prediction and loss; a **backward pass** computes gradients.
- PyTorch **autograd** computes the derivatives that we would otherwise calculate by hand.

**Our continuous example:** A learner studied `x = 2` hours and the target score is `5`. The toy model predicts `ŷ = w×x + b`, where we start with `w = 1`, `b = 1`.

The example intentionally uses one weight and one training point so you can check **every number**. A real model typically has many parameters and examples.


## 1. Derivatives: how sensitive is an output?

Start with the simple rule `f(x) = x²`.

At `x = 2`, the output is `f(2) = 4`. If we increase x a little, the output rises. A **derivative** measures the rate of change *at a particular input*. Geometrically, it is the slope of the tangent line to the curve.

For this rule, `f′(x) = 2x`. At `x = 2`, the derivative is **4**. That means a tiny increase of `0.01` in x changes f by roughly `4×0.01 = 0.04` near that point.

**The derivative is not the output.** At x=2 the output is 4 and the slope is also 4, but that numerical coincidence is specific to this example.


### What does a derivative answer?

The function answers **"What is the output right now?"** The derivative answers **"If I nudge the input slightly, how quickly does the output change?"** For `f(x)=x²`, at `x=2` the output is 4 and the derivative is 4. Near this point, increasing x by `0.01` raises f by approximately `4×0.01=0.04`.

**Why ML needs this:** a model adjusts weights rather than guessing endlessly. Derivatives describe how changing those weights will influence its error. The slope may be positive, negative, or zero, and is local: it need not stay the same for large changes.


<picture>
  <source type="image/svg+xml" srcset="images/13-derivative-tangent.svg">
  <img src="images/13-derivative-tangent.png" alt="The curve f(x)=x² and its tangent at x=2, highlighting that the derivative is its local slope" loading="lazy">
</picture>

*Figure: The curve f(x)=x² and its tangent at x=2, highlighting that the derivative is its local slope.*


In [ ]:
import torch

# Approximate the derivative from a small change in x.
def f(x):
    return x**2

x = 2.0
for h in [0.1, 0.01, 0.001]:
    approximate_derivative = (f(x+h)-f(x))/h
    print(f'h={h:5g} → approximate slope={approximate_derivative:.3f}')
print('Exact slope from 2*x:', 2*x)


### A gradient is several slopes collected together

When a loss depends on both `w` and `b`, we can ask two questions: how would the loss change if **only w** changed, and how would it change if **only b** changed? Those are partial derivatives. Combining them gives the **gradient** `∇L = [∂L/∂w, ∂L/∂b]`.

We calculate these values *at the current parameter settings*. A gradient says which small changes would initially raise or lower the loss; it is **not** a prediction or a parameter update.


As **h gets smaller**, the measured slope approaches 4. This is the idea behind the limit definition of a derivative. You do not need to derive all the calculus rules here; understanding what they *measure* is the important first step.

## 2. Partial derivatives and gradients

A prediction usually depends on several quantities, such as **weight `w`** and **bias `b`**. A **partial derivative** tells us how the result changes when one of these is changed while the others are held fixed.

For `ŷ = w×x + b`, treating x as a given input:

- `∂ŷ/∂w = x`: how the prediction changes with the weight.
- `∂ŷ/∂b = 1`: how the prediction changes with the bias.

If a loss depends on many parameters, we collect the partial derivatives into a **gradient**: `[∂L/∂w, ∂L/∂b, …]`. Each number tells us which way the loss would change if we adjusted its corresponding parameter.

## 3. First do the forward calculation

We want a prediction close to the target `5`. With `x=2`, `w=1` and `b=1`:

1. **Prediction:** `ŷ = 1×2 + 1 = 3`.
2. **Difference:** `ŷ - target = 3 - 5 = -2`.
3. **Squared-error loss:** `L = (-2)² = 4`.

A **loss function** converts the model's prediction error into a number that training tries to reduce. This simple squared error is not the only possible loss, but it makes the arithmetic easy to inspect.


<picture>
  <source type="image/svg+xml" srcset="images/14-forward-graph.svg">
  <img src="images/14-forward-graph.png" alt="Four connected stages of a forward pass: input, prediction, difference, and squared error" loading="lazy">
</picture>

*Figure: Four connected stages of a forward pass: input, prediction, difference, and squared error.*


In [ ]:
hours = torch.tensor(2.)
target = torch.tensor(5.)
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(1., requires_grad=True)

prediction = w*hours + b
loss = (prediction - target)**2

print('Prediction:',prediction.item())
print('Squared error:',loss.item())
print('Gradient BEFORE backward:',w.grad)
assert prediction.item()==3. and loss.item()==4.


Notice that `w.grad` is `None` here. The **forward pass** calculates the **value** of the prediction and loss, but it has not yet calculated the derivatives. PyTorch records operations involving `requires_grad=True` tensors so it can differentiate them later.

## 4. The chain rule explains the backward calculation

Our loss depends on `w` **through** the prediction:

\[w\;\longrightarrow\;\hat y = 2w+1\;\longrightarrow\;L=(\hat y-5)^2.\]

The **chain rule** says: multiply the slope of the loss with respect to the prediction by the slope of the prediction with respect to the weight:

\[\frac{dL}{dw} = \frac{dL}{d\hat y}\times\frac{d\hat y}{dw}.\]

At the starting point:

- `dL/dŷ = 2(ŷ - 5) = 2(3 - 5) = -4`.
- `dŷ/dw = x = 2`.
- So **`dL/dw = -4 × 2 = -8`**.

For bias, the slope is `dŷ/db = 1`, so `dL/db = -4 × 1 = -4`.

A *negative* derivative with respect to w means that **a small increase in w** would initially **reduce** the loss.


### Follow the complete chain backward

| Stage | Forward value | Local derivative needed for `dL/dw` |
|:--|:--|:--|
| Weight/input | `w=1`, `x=2`, `b=1` | `dŷ/dw=x=2` |
| Prediction | `ŷ=wx+b=3` | next: `dL/dŷ=2(ŷ−5)` |
| Difference | `ŷ−5=−2` | `2×(−2)=−4` |
| Squared loss | `L=(−2)²=4` | multiply `−4×2=−8` |

The loss gradient with respect to `w` is `−8`: increasing `w` slightly initially decreases loss. **The chain rule multiplies local derivatives along a path**, rather than multiplying outputs from successive forward passes. For the bias, `dŷ/db=1`, so `dL/db=−4`.


<picture>
  <source type="image/svg+xml" srcset="images/15-backward-chain-rule.svg">
  <img src="images/15-backward-chain-rule.png" alt="The chain rule calculation leading to gradient −8 and a weight update from 1 to 1.8" loading="lazy">
</picture>

*Figure: The chain rule calculation leading to gradient −8 and a weight update from 1 to 1.8.*


In [ ]:
loss.backward()  # run automatic differentiation
print('dL/dw:',w.grad.item())
print('dL/db:',b.grad.item())
assert w.grad.item()==-8.0
assert b.grad.item()==-4.0


### What did `backward()` actually do?

It **computed derivatives** and stored them in `.grad`. It did **not** update `w` or `b`. In neural-network training, computing gradients (backpropagation) and **using** those gradients to change parameters (optimization) are separate steps.

**One controlled update:** To see the effect of a gradient without changing two parameters simultaneously, we will update **only the weight** with learning rate `0.1`, holding b fixed at 1.

\[w_{new} = w_{old} - 0.1\frac{dL}{dw} = 1 - 0.1(-8) = 1.8.\]

A **learning rate** controls how large a step we take. The updated prediction is `2×1.8 + 1 = 4.6`, and squared error is `(4.6-5)² = 0.16`—smaller than 4.


### Distinguish three separate actions

1. **Forward propagation:** evaluate `ŷ = wx+b` and the loss, producing ordinary numerical values.
2. **Backward propagation (backpropagation):** traverse the recorded computation graph using the chain rule and accumulate derivatives in `.grad`.
3. **Optimizer step:** use those derivatives to **change** parameter values. A common update is `new_weight = old_weight − learning_rate × gradient`.

Autograd is the software system that computes derivatives; it does not decide the learning rate or update weights on its own. Forward-mode automatic differentiation is another derivative-computation method and is **not** just running the model's forward pass.


<picture>
  <source type="image/svg+xml" srcset="images/16-gradient-step.svg">
  <img src="images/16-gradient-step.png" alt="The loss curve showing how updating the weight from 1.0 to 1.8 reduces squared error from 4 to 0.16" loading="lazy">
</picture>

*Figure: The loss curve showing how updating the weight from 1.0 to 1.8 reduces squared error from 4 to 0.16.*


In [ ]:
learning_rate = 0.1
with torch.no_grad():  # this is an update, not a step we want to differentiate
    w -= learning_rate*w.grad

new_prediction = w*hours + b
new_loss = (new_prediction - target)**2
print('New weight:',round(w.item(),3))
print('New prediction:',round(new_prediction.item(),3))
print('New loss:',round(new_loss.item(),4))
assert abs(new_loss.item() - 0.16) < 1e-5


## 5. Why gradients need to be cleared

By default, PyTorch **accumulates** gradients into `.grad`: calling backward a second time can **add** derivatives rather than replacing the old ones. Training loops therefore typically clear gradients before the next backward pass (for example `optimizer.zero_grad()` or `w.grad.zero_()`).

The small demonstration below creates a **fresh weight** to avoid confusing its derivative with any previous calculation.


### Why can `.grad` surprise you?

Suppose one backward pass yields a gradient of `−8`. If we run another backward pass that contributes `−8`, PyTorch may leave `−16` in the same `.grad` buffer, because gradients **accumulate** by default. That is useful in some advanced training setups but misleading if you thought `.grad` would always be overwritten.

A normal training loop therefore clears gradients, calculates a new loss, calls `backward()`, and then updates parameters. **Clear → forward → backward → update** is a helpful beginner mental model.


In [ ]:
fresh_w = torch.tensor(1.,requires_grad=True)
first_loss = (2*fresh_w + 1 - 5)**2
first_loss.backward()
print('First gradient:',fresh_w.grad.item())

fresh_w.grad.zero_()
second_loss = (2*fresh_w + 1 - 5)**2
second_loss.backward()
print('After clearing, second gradient:',fresh_w.grad.item())


## 6. Commonly confused terms

| Term | What it means | What it **does not** mean |
|:--|:--|:--|
| **Forward propagation** | The model calculates its prediction (and loss) from current inputs/parameters. | It is not a parameter update. |
| **Automatic differentiation (autograd)** | Software calculates derivatives of code operations. | It is not the optimizer itself. |
| **Backpropagation** | A reverse pass applies the chain rule to calculate gradients of a loss with respect to parameters. | It does not automatically update weights. |
| **Gradient descent** | A procedure uses a gradient to change parameters, usually toward lower loss. | It is not the derivative calculation itself. |

**Forward-mode automatic differentiation** is another method for computing derivatives. It is **not the same thing** as “forward propagation.” We do not need forward-mode details for this beginner lesson.

## 7. Your turn: predict the derivative

If `L = (2w + 1 - 5)²`, what is `dL/dw` at `w = 2`? Check by hand **before** running the following cell. What does the derivative tell you about where the minimum might be?


In [ ]:
test_w = torch.tensor(2.,requires_grad=True)
test_loss = (2*test_w + 1 - 5)**2
test_loss.backward()
print('Loss at w=2:',test_loss.item())
print('Gradient at w=2:',test_w.grad.item())


### Check your understanding

1. Is a derivative the output of the function or the output's *rate of change*?
2. In the toy example, why is `w.grad` initially `None`?
3. What are the two multiplications involved in the **chain rule** that give `dL/dw = -8`?
4. Why do we write `with torch.no_grad():` around the manual update?
5. Does `.backward()` change parameters directly? If not, what changes them?

**Takeaway:** Training a model involves calculating a loss, finding how sensitive it is to the parameters (gradients), and applying updates that aim to reduce that loss. This is the beginning of **backpropagation**, the mechanism used to train neural networks.

➡️ **Next:** [2.5 Probability and statistics](05-probability-and-statistics.ipynb) — why individual outcomes and datasets are uncertain.
